# AI-CFD DGCNN Training — Colab Complete Resume Version

이 노트북은 런타임 초기화에 대비해 처음부터 끝까지 다시 정리한 완성본이다.

현재 학습 정의:

- Input: `[x, y, z, velocity]`
- Target: `[HTC, wall_shear, pressure]`
- DGCNN regression head: `512 → 256 → 128 → 3`
- Trainable parameters: `189,955`
- FPS: `7000 points/sample`
- `k = 20`

Google Drive에 아래 파일이 있어야 한다.

- `MyDrive/ai-cfd-flow-prediction/data/05_cfd_csv.zip`
- `MyDrive/ai-cfd-flow-prediction/data/fps_indices_7000.npz`

학습 결과는 자동으로 아래에 저장된다.

- `MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt`
- `MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz`
- `MyDrive/ai-cfd-flow-prediction/dgcnn/last_checkpoint.pt`

런타임이 초기화되면 Cell 1 → 2 → 3 → 5 순서로 다시 실행하면 된다.

Cell 5는 기존 결과를 무조건 resume하지 않는다.

- `output_dim = 3`
- `point_count = 7000`
- `k = 20`
- `first_knn_space = "raw_xyz"`
- 3-target scaler

가 모두 일치할 때만 resume한다.

기존 `[HTC, wall_shear]` 2-output checkpoint가 남아 있으면
호환되지 않는 것으로 판정하고 새 3-target 학습을 시작한다.


## Cell 1 — GPU 확인


In [ ]:
!nvidia-smi

import torch

print()
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 잡히지 않았습니다. "
        "Colab에서 런타임 유형을 T4 GPU로 변경하세요."
    )

print("GPU            :", torch.cuda.get_device_name(0))
print(
    "GPU memory     :",
    torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "GB",
)


## Cell 2 — Google Drive 연결 및 영구 경로 설정


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction"
)

DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_DGCNN = DRIVE_ROOT / "dgcnn"

DRIVE_ZIP = DRIVE_DATA / "05_cfd_csv.zip"
DRIVE_FPS = DRIVE_DATA / "fps_indices_7000.npz"

BEST_MODEL = DRIVE_DGCNN / "best_model.pt"
SCALER = DRIVE_DGCNN / "scalers.npz"
LAST_CHECKPOINT = DRIVE_DGCNN / "last_checkpoint.pt"
TEST_LOG = DRIVE_DGCNN / "test_evaluation.txt"

DRIVE_DGCNN.mkdir(
    parents=True,
    exist_ok=True,
)

required_files = [
    DRIVE_ZIP,
    DRIVE_FPS,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Google Drive 파일을 찾을 수 없습니다:\n{path}"
        )

print("=" * 78)
print("GOOGLE DRIVE READY")
print("=" * 78)
print("CSV ZIP         :", DRIVE_ZIP)
print("FPS index       :", DRIVE_FPS)
print("DGCNN output    :", DRIVE_DGCNN)
print("Best model      :", BEST_MODEL)
print("Scaler          :", SCALER)
print("Last checkpoint :", LAST_CHECKPOINT)
print("=" * 78)


## Cell 3 — GitHub 코드 및 학습 데이터 자동 복구


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DATA_ROOT = Path(
    "/content/ai-cfd-data"
)

CSV_DIR = (
    DATA_ROOT
    / "05_cfd_csv"
)

FPS_DST = (
    REPO
    / "04_cfd_dataset"
    / "fps_indices_7000.npz"
)

# ================================================================
# 1. GitHub repository
# ================================================================

if REPO.exists():

    print(
        "[Git] Existing repository found -> pull latest main",
        flush=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "pull",
            "origin",
            "main",
        ],
        check=True,
    )

else:

    print(
        "[Git] Repository not found -> clone",
        flush=True,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/hehong01/ai-cfd-flow-prediction.git",
            str(REPO),
        ],
        check=True,
    )

# ================================================================
# 2. CFD CSV restore
# ================================================================

def count_direct_csv():
    if not CSV_DIR.exists():
        return 0

    return len(
        list(
            CSV_DIR.glob("*.csv")
        )
    )

csv_count = count_direct_csv()

if csv_count != 300:

    print(
        f"[Data] Local CSV count = {csv_count} -> restore from Drive",
        flush=True,
    )

    shutil.rmtree(
        DATA_ROOT,
        ignore_errors=True,
    )

    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_zip = Path(
        "/content/05_cfd_csv.zip"
    )

    shutil.copy2(
        DRIVE_ZIP,
        local_zip,
    )

    # ZIP 내부에 05_cfd_csv/ 폴더가 있으므로
    # /content/ai-cfd-data 에 직접 압축 해제한다.
    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(local_zip),
            "-d",
            str(DATA_ROOT),
        ],
        check=True,
    )

    csv_count = count_direct_csv()

else:

    print(
        "[Data] Existing 300 local CSV files -> reuse",
        flush=True,
    )

if csv_count != 300:
    raise RuntimeError(
        "CFD CSV 복구 실패:\n"
        f"Expected 300, found {csv_count}\n"
        f"Directory: {CSV_DIR}"
    )

# ================================================================
# 3. FPS index restore
# ================================================================

FPS_DST.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    DRIVE_FPS,
    FPS_DST,
)

if not FPS_DST.exists():
    raise RuntimeError(
        f"FPS 복사 실패:\n{FPS_DST}"
    )

# ================================================================
# 4. State check
# ================================================================

commit = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "--short",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)

print()
print("=" * 78)
print("COLAB ENVIRONMENT READY")
print("=" * 78)
print("Git commit       :", commit)
print("CSV count        :", csv_count)
print("CSV directory    :", CSV_DIR)
print("FPS exists       :", FPS_DST.exists())
print("FPS path         :", FPS_DST)
print(
    "Checkpoint exists:",
    LAST_CHECKPOINT.exists(),
)
print("=" * 78)


## Cell 4 — Dataset 전체 검증

처음 세팅하거나 데이터 경로가 의심될 때 실행한다.

정상이라면 다음이 모두 확인되어야 한다.

- TRAIN: 240 samples
- VAL: 30 samples
- TEST: 30 samples
- TOTAL: 300 samples
- 7000 points/sample
- X shape: `(7000, 4)`
- Y shape: `(7000, 3)`
- Target: `[HTC, wall_shear, pressure]`


In [ ]:
%cd /content/ai-cfd-flow-prediction
!python -u 04_cfd_dataset/dataset.py


## Cell 5 — DGCNN 본학습 / 자동 Resume / 실시간 로그

이 셀은 완성본이다.

- `last_checkpoint.pt`가 없으면 새 3-target 학습 시작
- checkpoint가 있어도 `output_dim=3`, `7000 points`, `k=20`, `raw_xyz`가 맞는지 검사
- `best_model.pt`와 `scalers.npz`도 같은 3-target 형식인지 검사
- 기존 2-output 결과이면 resume하지 않고 `--overwrite`로 새 학습 시작
- 호환되는 3-target 결과일 때만 저장된 다음 epoch부터 자동 resume
- `train.py`의 stdout/stderr를 한 줄씩 직접 읽어서 Colab에 실시간 출력
- 매 epoch 종료 후 `last_checkpoint.pt`가 Google Drive에 갱신
- best validation model은 `best_model.pt`에 유지


In [ ]:
# ================================================================
# Cell 5 — DGCNN full training / automatic resume / live logging
# ================================================================

from pathlib import Path
import subprocess
import sys
import numpy as np
import torch

# ================================================================
# Paths
# ================================================================

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction"
)

DRIVE_DGCNN = (
    DRIVE_ROOT
    / "dgcnn"
)

DRIVE_DGCNN.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_SCRIPT = (
    REPO
    / "05_model_training"
    / "dgcnn"
    / "train.py"
)

BEST_MODEL = (
    DRIVE_DGCNN
    / "best_model.pt"
)

SCALER = (
    DRIVE_DGCNN
    / "scalers.npz"
)

LAST_CHECKPOINT = (
    DRIVE_DGCNN
    / "last_checkpoint.pt"
)

# ================================================================
# Final training settings
# ================================================================

TARGET_EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 16

EXPECTED_MODEL_NAME = "DGCNNRegressor"
EXPECTED_OUTPUT_DIM = 3
EXPECTED_POINT_COUNT = 7000
EXPECTED_K = 20
EXPECTED_FIRST_KNN_SPACE = "raw_xyz"
EXPECTED_PARAMETER_COUNT = 189_955

# ================================================================
# Basic checks
# ================================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU가 잡히지 않았습니다. "
        "Colab 런타임을 T4 GPU로 변경하세요."
    )

if not TRAIN_SCRIPT.exists():

    raise FileNotFoundError(
        "train.py를 찾을 수 없습니다:\n"
        f"{TRAIN_SCRIPT}"
    )

print(
    "=" * 78,
    flush=True,
)

print(
    "DGCNN FINAL TRAINING LAUNCHER",
    flush=True,
)

print(
    "=" * 78,
    flush=True,
)

print(
    f"GPU             : {torch.cuda.get_device_name(0)}",
    flush=True,
)

print(
    f"Batch size      : {BATCH_SIZE}",
    flush=True,
)

print(
    f"Target epochs   : {TARGET_EPOCHS}",
    flush=True,
)

print(
    f"Patience        : {PATIENCE}",
    flush=True,
)

print(
    "Targets         : [HTC, wall_shear, pressure]",
    flush=True,
)

print(
    "Architecture    : 512 -> 256 -> 128 -> 3",
    flush=True,
)

print(
    f"Parameter target: {EXPECTED_PARAMETER_COUNT:,}",
    flush=True,
)

print(
    f"Best model      : {BEST_MODEL}",
    flush=True,
)

print(
    f"Scaler          : {SCALER}",
    flush=True,
)

print(
    f"Last checkpoint : {LAST_CHECKPOINT}",
    flush=True,
)

print(
    "=" * 78,
    flush=True,
)

print(
    flush=True,
)

# ================================================================
# Fresh training vs resume
# ================================================================

run_training = True
mode = None

if LAST_CHECKPOINT.exists():

    print(
        "Existing last_checkpoint.pt detected.",
        flush=True,
    )

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    checkpoint_model_name = str(
        checkpoint.get(
            "model_name",
            "",
        )
    )

    checkpoint_output_dim = int(
        checkpoint.get(
            "output_dim",
            -1,
        )
    )

    checkpoint_point_count = int(
        checkpoint.get(
            "point_count",
            -1,
        )
    )

    checkpoint_k = int(
        checkpoint.get(
            "k",
            -1,
        )
    )

    checkpoint_first_knn_space = str(
        checkpoint.get(
            "first_knn_space",
            "",
        )
    )

    compatible = (
        checkpoint_model_name == EXPECTED_MODEL_NAME
        and checkpoint_output_dim == EXPECTED_OUTPUT_DIM
        and checkpoint_point_count == EXPECTED_POINT_COUNT
        and checkpoint_k == EXPECTED_K
        and checkpoint_first_knn_space
        == EXPECTED_FIRST_KNN_SPACE
    )

    if not compatible:

        print()
        print(
            "[INCOMPATIBLE OLD CHECKPOINT]",
            flush=True,
        )
        print(
            f"Model name      : {checkpoint_model_name!r}",
            flush=True,
        )
        print(
            f"Output dim      : {checkpoint_output_dim}",
            flush=True,
        )
        print(
            f"Point count     : {checkpoint_point_count}",
            flush=True,
        )
        print(
            f"k               : {checkpoint_k}",
            flush=True,
        )
        print(
            f"First kNN space : {checkpoint_first_knn_space!r}",
            flush=True,
        )

        print()
        print(
            "기존 2-output 또는 호환되지 않는 DGCNN 결과는 "
            "resume하지 않습니다.",
            flush=True,
        )
        print(
            "새 3-target DGCNN을 epoch 1부터 학습하고 "
            "Drive의 DGCNN 결과를 새 결과로 덮어씁니다.",
            flush=True,
        )

        mode = "--overwrite"

    else:

        if not SCALER.exists():

            raise RuntimeError(
                "호환되는 3-target last_checkpoint.pt는 존재하지만 "
                "scalers.npz가 없습니다.\n"
                "Resume할 수 없는 불완전한 상태입니다."
            )

        if not BEST_MODEL.exists():

            raise RuntimeError(
                "호환되는 3-target last_checkpoint.pt는 존재하지만 "
                "best_model.pt가 없습니다.\n"
                "Resume할 수 없는 불완전한 상태입니다."
            )

        best_checkpoint = torch.load(
            BEST_MODEL,
            map_location="cpu",
            weights_only=False,
        )

        best_model_name = str(
            best_checkpoint.get(
                "model_name",
                "",
            )
        )

        best_output_dim = int(
            best_checkpoint.get(
                "output_dim",
                -1,
            )
        )

        best_point_count = int(
            best_checkpoint.get(
                "point_count",
                -1,
            )
        )

        best_k = int(
            best_checkpoint.get(
                "k",
                -1,
            )
        )

        best_first_knn_space = str(
            best_checkpoint.get(
                "first_knn_space",
                "",
            )
        )

        best_compatible = (
            best_model_name == EXPECTED_MODEL_NAME
            and best_output_dim == EXPECTED_OUTPUT_DIM
            and best_point_count == EXPECTED_POINT_COUNT
            and best_k == EXPECTED_K
            and best_first_knn_space
            == EXPECTED_FIRST_KNN_SPACE
        )

        if not best_compatible:

            raise RuntimeError(
                "last_checkpoint.pt는 3-target 형식이지만 "
                "best_model.pt가 호환되지 않습니다.\n"
                f"Best model name      : {best_model_name!r}\n"
                f"Best output dim      : {best_output_dim}\n"
                f"Best point count     : {best_point_count}\n"
                f"Best k               : {best_k}\n"
                f"Best first kNN space : {best_first_knn_space!r}"
            )

        with np.load(
            SCALER
        ) as scaler_data:

            for key in (
                "input_mean",
                "input_std",
                "target_mean",
                "target_std",
            ):

                if key not in scaler_data.files:

                    raise RuntimeError(
                        f"scalers.npz에 {key}가 없습니다."
                    )

            input_mean_shape = tuple(
                scaler_data["input_mean"].shape
            )

            input_std_shape = tuple(
                scaler_data["input_std"].shape
            )

            target_mean_shape = tuple(
                scaler_data["target_mean"].shape
            )

            target_std_shape = tuple(
                scaler_data["target_std"].shape
            )

        if (
            input_mean_shape != (4,)
            or input_std_shape != (4,)
            or target_mean_shape != (EXPECTED_OUTPUT_DIM,)
            or target_std_shape != (EXPECTED_OUTPUT_DIM,)
        ):

            raise RuntimeError(
                "scalers.npz가 현재 3-target DGCNN 형식이 아닙니다.\n"
                f"input_mean shape : {input_mean_shape}\n"
                f"input_std shape  : {input_std_shape}\n"
                f"target_mean shape: {target_mean_shape}\n"
                f"target_std shape : {target_std_shape}"
            )

        required_keys = [
            "epoch",
            "best_epoch",
            "best_val_loss",
            "epochs_without_improvement",
        ]

        for key in required_keys:

            if key not in checkpoint:

                raise KeyError(
                    "Resume checkpoint에 필수 항목이 없습니다: "
                    f"{key}"
                )

        completed_epoch = int(
            checkpoint["epoch"]
        )

        best_epoch = int(
            checkpoint["best_epoch"]
        )

        best_val_loss = float(
            checkpoint["best_val_loss"]
        )

        no_improve_count = int(
            checkpoint[
                "epochs_without_improvement"
            ]
        )

        print()
        print(
            "[COMPATIBLE 3-TARGET DGCNN CHECKPOINT]",
            flush=True,
        )
        print(
            f"Completed epoch  : {completed_epoch}",
            flush=True,
        )
        print(
            f"Best epoch       : {best_epoch}",
            flush=True,
        )
        print(
            f"Best val loss    : {best_val_loss:.8f}",
            flush=True,
        )
        print(
            f"No-improve count : {no_improve_count}",
            flush=True,
        )
        print(
            f"Output dim       : {checkpoint_output_dim}",
            flush=True,
        )
        print(
            f"Point count      : {checkpoint_point_count}",
            flush=True,
        )
        print(
            f"k                : {checkpoint_k}",
            flush=True,
        )
        print(
            f"Scaler targets   : {target_mean_shape}",
            flush=True,
        )
        print()

        if completed_epoch >= TARGET_EPOCHS:

            print(
                "Training is already complete "
                f"({completed_epoch}/{TARGET_EPOCHS}).",
                flush=True,
            )

            run_training = False

        elif no_improve_count >= PATIENCE:

            print(
                "Training already reached the "
                "early-stopping condition.",
                flush=True,
            )

            run_training = False

        else:

            mode = "--resume"

            print(
                "RESUME TRAINING: "
                f"epoch {completed_epoch + 1}부터 시작합니다.",
                flush=True,
            )

else:

    mode = "--overwrite"

    print(
        "No resume checkpoint -> "
        "새 3-target DGCNN 본학습을 시작합니다.",
        flush=True,
    )

print(
    flush=True,
)

# ================================================================
# Launch train.py
# ================================================================

if run_training:

    cmd = [
        sys.executable,
        "-u",
        str(TRAIN_SCRIPT),

        "--batch-size",
        str(BATCH_SIZE),

        "--epochs",
        str(TARGET_EPOCHS),

        "--patience",
        str(PATIENCE),

        "--model-path",
        str(BEST_MODEL),

        "--scaler-path",
        str(SCALER),

        "--checkpoint-path",
        str(LAST_CHECKPOINT),

        mode,
    ]

    print(
        "=" * 78,
        flush=True,
    )

    print(
        "COMMAND",
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )

    print(
        " ".join(cmd),
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )

    print(
        flush=True,
    )

    # ------------------------------------------------------------
    # Start child process.
    #
    # stdout + stderr를 하나의 PIPE로 받고,
    # 한 줄씩 즉시 Colab 출력창에 전달한다.
    # ------------------------------------------------------------

    process = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:

        process.kill()

        raise RuntimeError(
            "train.py stdout pipe를 열 수 없습니다."
        )

    try:

        for line in iter(
            process.stdout.readline,
            "",
        ):

            if line == "":
                break

            print(
                line,
                end="",
                flush=True,
            )

    finally:

        process.stdout.close()

    return_code = process.wait()

    if return_code != 0:

        raise RuntimeError(
            "DGCNN training failed "
            f"(exit code {return_code})."
        )

    print(
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )

    print(
        "DGCNN TRAINING PROCESS FINISHED",
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )


## Cell 6 — 최종 held-out TEST 평가

학습이 끝난 뒤 한 번 실행한다.

- test split: face_0091 ~ face_0100
- 30 samples
- full 7000 points/sample
- 결과는 Drive의 `dgcnn/test_evaluation.txt`에도 저장


In [ ]:
%cd /content/ai-cfd-flow-prediction

!python -u 05_model_training/dgcnn/evaluate.py \
    --batch-size 16 \
    --model-path "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt" \
    --scaler-path "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz" \
    | tee "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/test_evaluation.txt"


## Cell 7 — Drive 결과 파일 확인


In [ ]:
from pathlib import Path

DRIVE_DGCNN = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn"
)

print("=" * 78)
print("DGCNN DRIVE OUTPUTS")
print("=" * 78)

if not DRIVE_DGCNN.exists():

    raise FileNotFoundError(
        f"DGCNN Drive 폴더가 없습니다:\n{DRIVE_DGCNN}"
    )

for path in sorted(
    DRIVE_DGCNN.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:24s} "
            f"{path.stat().st_size / 1024**2:8.2f} MB"
        )

print("=" * 78)


Cell 8 — 최종 성능 평가


In [ ]:
# ================================================================
# FINAL DGCNN RESULT SUMMARY
# ================================================================

from pathlib import Path
import numpy as np
import torch

DRIVE_DGCNN = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn"
)

BEST_MODEL = DRIVE_DGCNN / "best_model.pt"
LAST_CHECKPOINT = DRIVE_DGCNN / "last_checkpoint.pt"
SCALER = DRIVE_DGCNN / "scalers.npz"
TEST_LOG = DRIVE_DGCNN / "test_evaluation.txt"

EXPECTED_MODEL_NAME = "DGCNNRegressor"
EXPECTED_OUTPUT_DIM = 3
EXPECTED_POINT_COUNT = 7000
EXPECTED_K = 20
EXPECTED_FIRST_KNN_SPACE = "raw_xyz"

print("=" * 78)
print("FINAL DGCNN RESULT SUMMARY")
print("=" * 78)

# ------------------------------------------------
# Best model metadata
# ------------------------------------------------

if not BEST_MODEL.exists():
    raise FileNotFoundError(BEST_MODEL)

best = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

best_model_name = str(
    best.get("model_name", "")
)

best_output_dim = int(
    best.get("output_dim", -1)
)

best_point_count = int(
    best.get("point_count", -1)
)

best_k = int(
    best.get("k", -1)
)

best_first_knn_space = str(
    best.get("first_knn_space", "")
)

if (
    best_model_name != EXPECTED_MODEL_NAME
    or best_output_dim != EXPECTED_OUTPUT_DIM
    or best_point_count != EXPECTED_POINT_COUNT
    or best_k != EXPECTED_K
    or best_first_knn_space != EXPECTED_FIRST_KNN_SPACE
):

    raise RuntimeError(
        "best_model.pt가 현재 3-target DGCNN 형식이 아닙니다.\n"
        f"model_name      : {best_model_name!r}\n"
        f"output_dim      : {best_output_dim}\n"
        f"point_count     : {best_point_count}\n"
        f"k               : {best_k}\n"
        f"first_knn_space : {best_first_knn_space!r}"
    )

print()
print("[BEST MODEL]")
print("Best epoch       :", best.get("epoch"))
print("Best val loss    :", best.get("val_loss"))
print("Output dim       :", best_output_dim)
print("Targets          :", "[HTC, wall_shear, pressure]")
print("Points/sample    :", best_point_count)
print("k                :", best_k)
print("First kNN space  :", best_first_knn_space)

# ------------------------------------------------
# Last training checkpoint
# ------------------------------------------------

if not LAST_CHECKPOINT.exists():
    raise FileNotFoundError(LAST_CHECKPOINT)

last = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

last_model_name = str(
    last.get("model_name", "")
)

last_output_dim = int(
    last.get("output_dim", -1)
)

last_point_count = int(
    last.get("point_count", -1)
)

last_k = int(
    last.get("k", -1)
)

last_first_knn_space = str(
    last.get("first_knn_space", "")
)

if (
    last_model_name != EXPECTED_MODEL_NAME
    or last_output_dim != EXPECTED_OUTPUT_DIM
    or last_point_count != EXPECTED_POINT_COUNT
    or last_k != EXPECTED_K
    or last_first_knn_space != EXPECTED_FIRST_KNN_SPACE
):

    raise RuntimeError(
        "last_checkpoint.pt가 현재 3-target DGCNN 형식이 아닙니다.\n"
        f"model_name      : {last_model_name!r}\n"
        f"output_dim      : {last_output_dim}\n"
        f"point_count     : {last_point_count}\n"
        f"k               : {last_k}\n"
        f"first_knn_space : {last_first_knn_space!r}"
    )

print()
print("[LAST TRAINING STATE]")
print("Completed epoch  :", last.get("epoch"))
print("Best epoch       :", last.get("best_epoch"))
print("Best val loss    :", last.get("best_val_loss"))
print(
    "No-improve count :",
    last.get("epochs_without_improvement"),
)
print("Output dim       :", last_output_dim)
print("Points/sample    :", last_point_count)
print("k                :", last_k)

# ------------------------------------------------
# Scaler validation
# ------------------------------------------------

if not SCALER.exists():
    raise FileNotFoundError(SCALER)

with np.load(
    SCALER
) as scaler_data:

    input_mean_shape = tuple(
        scaler_data["input_mean"].shape
    )

    input_std_shape = tuple(
        scaler_data["input_std"].shape
    )

    target_mean_shape = tuple(
        scaler_data["target_mean"].shape
    )

    target_std_shape = tuple(
        scaler_data["target_std"].shape
    )

if (
    input_mean_shape != (4,)
    or input_std_shape != (4,)
    or target_mean_shape != (EXPECTED_OUTPUT_DIM,)
    or target_std_shape != (EXPECTED_OUTPUT_DIM,)
):

    raise RuntimeError(
        "scalers.npz가 현재 3-target DGCNN 형식이 아닙니다.\n"
        f"input_mean shape : {input_mean_shape}\n"
        f"input_std shape  : {input_std_shape}\n"
        f"target_mean shape: {target_mean_shape}\n"
        f"target_std shape : {target_std_shape}"
    )

print()
print("[SCALER]")
print("Input features   :", input_mean_shape)
print("Target features  :", target_mean_shape)
print("Target order     :", "[HTC, wall_shear, pressure]")

# ------------------------------------------------
# File sizes
# ------------------------------------------------

print()
print("[FILES]")

for path in (
    BEST_MODEL,
    LAST_CHECKPOINT,
    SCALER,
    TEST_LOG,
):
    print(
        f"{path.name:24s} "
        f"{path.stat().st_size:,} bytes"
    )

# ------------------------------------------------
# Held-out TEST result
# ------------------------------------------------

print()
print("=" * 78)
print("HELD-OUT TEST RESULT")
print("=" * 78)

if not TEST_LOG.exists():
    raise FileNotFoundError(TEST_LOG)

test_text = TEST_LOG.read_text(
    encoding="utf-8",
    errors="replace",
)

print(test_text)

print("=" * 78)